In [1]:
import pandas as pd

# Baca data hasil merge yang sudah memiliki kolom Dominant_Topic per game
df = pd.read_csv("combined_player_game_with_topics.csv")

# Tampilkan 5 baris pertama
print("Data gabungan combined_player_game_with_topics.csv:")
display(df.head())


Data gabungan combined_player_game_with_topics.csv:


,Steam ID,App ID,Game Name,Playtime (hours),Genres,Achievements
0,76561198881119093,20,team fortress classic,0.316667,Action,0
1,76561198881119093,50,half-life: opposing force,7.600000,Action,0
2,76561198881119093,70,half-life,18.166667,Action,0
3,76561198881119093,130,half-life: blue shift,2.783333,Action,0
4,76561198881119093,220,half-life 2,20.550000,Action,35


In [4]:
# Jumlah baris awal
initial_rows = len(df)
print(f"Jumlah baris awal: {initial_rows}")

# Hapus baris dengan nilai NaN di kolom-kolom penting
required_columns = ["Steam ID", "App ID", "Game Name", "Playtime (hours)", "Genres", "Achievements"]
df = df.dropna(subset=required_columns)

# Jumlah baris setelah pembersihan
cleaned_rows = len(df)
print(f"Jumlah baris setelah menghapus NaN dan kolom 'Similarity_Score': {cleaned_rows}")


Jumlah baris awal: 103206
Jumlah baris setelah menghapus NaN dan kolom 'Similarity_Score': 100100


In [5]:
# Simpan jumlah baris sebelum penghapusan game nonaktif
before_removal = len(df)

# Filter: hanya ambil game dengan playtime >= 10 menit (yaitu >= 0.166 jam)
df = df[df["Playtime (hours)"] >= 0.166]

# Jumlah baris setelah penghapusan
after_removal = len(df)

print(f"Jumlah baris sebelum penghapusan game nonaktif: {before_removal}")
print(f"Jumlah baris setelah penghapusan game nonaktif: {after_removal}")


Jumlah baris sebelum penghapusan game nonaktif: 100100
Jumlah baris setelah penghapusan game nonaktif: 50842


In [6]:
# Simpan jumlah baris sebelum penghapusan
before_removal = len(df)

# Hapus game dengan nama 'Unknown'
df = df[df["Game Name"].str.lower() != "unknown"]

# Simpan jumlah baris setelah penghapusan
after_removal = len(df)

print(f"Jumlah baris sebelum penghapusan game 'Unknown': {before_removal}")
print(f"Jumlah baris setelah penghapusan game 'Unknown': {after_removal}")


Jumlah baris sebelum penghapusan game 'Unknown': 50842
Jumlah baris setelah penghapusan game 'Unknown': 50837


In [7]:
# Simpan jumlah baris sebelum penghapusan
before_dedup = len(df)

# Hapus data duplikat
df = df.drop_duplicates()

# Simpan jumlah baris setelah penghapusan
after_dedup = len(df)

print(f"Jumlah baris sebelum penghapusan duplikat: {before_dedup}")
print(f"Jumlah baris setelah penghapusan duplikat: {after_dedup}")


Jumlah baris sebelum penghapusan duplikat: 50837
Jumlah baris setelah penghapusan duplikat: 50837


# Transformation

In [8]:
# Hitung jumlah game per SteamID
game_counts = df['Steam ID'].value_counts()

# Tambahkan kolom baru 'Game' berdasarkan jumlah game dari masing-masing SteamID
df['Game'] = df['Steam ID'].map(game_counts)

# Cek hasil
df[['Steam ID', 'Game']].drop_duplicates().head()


,Steam ID,Game
0,76561198881119093,20
27,76561198151246013,137
388,76561198068123705,267
1431,76561198355865927,77
1520,76561198095282532,53


In [9]:
# Hitung Q1 (kuartil pertama) dari jumlah game
q1 = df['Game'].quantile(0.25)

# Tampilkan Q1 untuk informasi
print(f"Q1 Jumlah Game: {q1}")

# Filter data: hanya ambil pemain yang jumlah game-nya >= Q1
df = df[df['Game'] >= q1].reset_index(drop=True)

# Cek jumlah data setelah filter
df['Steam ID'].nunique(), df.shape


Q1 Jumlah Game: 212.0


(79, (38275, 7))

# Ubah Value Genre

In [10]:
# 1. Pecah genre menjadi list
df['Genre_List'] = df['Genres'].fillna('').apply(lambda x: [g.strip() for g in x.split(',') if g.strip() != ''])

# 2. Ambil semua genre unik dari seluruh list
from itertools import chain

all_genres = set(chain.from_iterable(df['Genre_List']))
genre_mapping = {genre: i+1 for i, genre in enumerate(sorted(all_genres))}

# 3. Ganti list genre jadi list kode
df['Genre_Code_List'] = df['Genre_List'].apply(lambda genre_list: [genre_mapping[g] for g in genre_list if g in genre_mapping])

# 4. Simpan mapping ke CSV
genre_df = pd.DataFrame(list(genre_mapping.items()), columns=['Genre', 'Genre_Code'])
genre_df.to_csv('genre_code_mapping.csv', index=False)

# 5. Simpan data utama
df.to_csv('cleaned_with_genre_codes.csv', index=False)


# Ubah Nilai Fitur (FIX)

In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Load dan bersihkan header
df = pd.read_csv('transformation_segmentation_v4.csv')
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(' ', '_')
)
# Sekarang df.columns → ['no', 'steam_id', 'total_game',
#                         'total_achievement', 'total_playtime',
#                         'topic_dominan', 'genre_dominan']

# 2. Pilih kolom dasar sebagai user_base
#    steam_id, total_game, total_achievement, total_playtime
user_base = (
    df[['steam_id','total_game','total_achievement','total_playtime']]
    .drop_duplicates(subset='steam_id')
    .set_index('steam_id')
)

# 3. Pivot frekuensi topik & genre per user
topic_pivot = (
    df.groupby(['steam_id','topic_dominan'])
      .size()
      .unstack(fill_value=0)
      .add_prefix('topic_count_')
)
genre_pivot = (
    df.groupby(['steam_id','genre_dominan'])
      .size()
      .unstack(fill_value=0)
      .add_prefix('genre_count_')
)

# 4. Gabungkan semua fitur ke satu DataFrame
user_features = (
    user_base
    .join(topic_pivot, how='left').fillna(0)
    .join(genre_pivot, how='left').fillna(0)
)

# 5. Hitung proporsi tiap count relatif ke total_game
count_cols = [c for c in user_features.columns if c.startswith(('topic_count_','genre_count_'))]
for c in count_cols:
    user_features[c + '_prop'] = user_features[c] / user_features['total_game']

# 6. Siapkan daftar kolom untuk normalisasi
norm_cols = (
    count_cols +
    [c + '_prop' for c in count_cols] +
    ['total_game','total_achievement','total_playtime']
)

# 7. Konversi ke float & normalisasi MinMax
user_features[norm_cols] = user_features[norm_cols].astype(float)
scaler = MinMaxScaler()
user_features[norm_cols] = scaler.fit_transform(user_features[norm_cols])

# 8. Reset index & simpan hasil
user_features = user_features.reset_index()
user_features.to_csv('user_features_for_AA_v2.csv', index=False)

print("✅ Selesai. user_features_for_AA_v2.csv shape:", user_features.shape)


✅ Selesai. user_features_for_AA_v2.csv shape: (59, 18)


In [8]:
import pandas as pd

# Load the two input files
df_v4   = pd.read_csv('transformation_segmentation_v4.csv')
df_seg  = pd.read_csv('sampled_segmentation.csv')

# Print all column names from each
print("Columns in 'transformation_segmentation_v4.csv':")
for col in df_v4.columns:
    print(f" - {col}")

print("\nColumns in 'sampled_segmentation.csv':")
for col in df_seg.columns:
    print(f" - {col}")


Columns in 'transformation_segmentation_v4.csv':
 - No
 - Steam ID
 - Total Game
 - Total Achievement
 - Total Playtime
 - Topic Dominan
 - Genre Dominan

Columns in 'sampled_segmentation.csv':
 - Steam ID
 - App ID
 - Game Name
 - Playtime (hours)
 - Genres
 - Achievements
